In [ ]:
from openai import OpenAI
import os
import json

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [ ]:
client = OpenAI(
   
    api_key=OPENAI_API_KEY,
)

CHAT_MODEL = "gpt-3.5-turbo"
PLANNER_MODEL = "gpt-4o"
IMAGE_MODEL = "gpt-image-1"
IMAGE_QUALITY = "medium"
DAY_IMAGE_QUALITY = "low"


PRICES = {
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
    "gpt-4o":        {"input": 2.50, "output": 10.00},
    "gpt-image-1":   {"input": 5.00, "output": 40.00},
}

USAGE = []  # one record per API call; cleared when a new conversation starts


def get_completion_from_messages(messages, model=CHAT_MODEL, temperature=0): 
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    u = response.usage
    USAGE.append({
        "model": model,
        "input": u.prompt_tokens,
        "output": u.completion_tokens,
    })
    return response.choices[0].message.content

In [ ]:
# cell 3
USAGE.clear()  

context = [{'role': 'system', 'content': """
You are a friendly travel assistant. Your ONLY job in this conversation is to
collect information. You do NOT write the itinerary — another step does that.

Greet the traveler warmly, then collect, in this order:
1. Destination, number of days, and the month.
2. Main interests (food, museums, hiking, beaches, nightlife, shopping...).
3. Constraints: mobility, allergies, diet, travelling with kids, etc.

Rules:
- Ask about ONE topic per message, and keep messages short and friendly.
- Never ask a question they have already answered. Move on.
- If an answer is ambiguous or contradicts something earlier, ask one
  clarifying question before continuing.
- The trip must be organised around CITIES OR TOWNS the traveler sleeps in, never
  regions, provinces or countries. If they name a region or country ("Galicia",
  "Tuscany", "Japan"), say it is a big area and ask which one or two cities they
  want to use as bases, then split the nights between those cities.
- If they name more than one city, ask how many nights in each and how they plan
  to travel between them.
- If the nights per city do not sum to the stated trip length, ask which is
  correct before recapping. Do not silently pick one.
- If the traveler is unsure or asks for suggestions, reassure them briefly
  ("no worries — here's what most people love in <destination>"), then offer
  3 or 4 concrete options for that destination and let them pick. Never record a
  topic as "not specified".
- Once you have all three topics, stop asking. Output RECAP: followed by a
  single-line JSON object with keys destination, country (the English name of the
  country the cities are in, e.g. "Spain"), days (integer — total trip length),
  nights_per_city (object mapping each CITY OR TOWN → number of nights),
  inter_city_transport (string, e.g. "car", "train"), month, interests,
  constraints.
  Then the word READY on its own line, and nothing after it.
"""}]

In [ ]:
# cell 4
while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", ""):
        break
    print(f"You: {user_input}")
    context.append({'role': 'user', 'content': user_input})
    response = get_completion_from_messages(context, temperature=0)
    context.append({'role': 'assistant', 'content': response})
    print(f"TravelBot: {response}\n{'-'*60}")
    if response.strip().endswith("READY"):
        break


In [ ]:
# cell 5
if "RECAP:" not in response:
    raise ValueError(
        "The last reply from cell 4 has no RECAP: line, so there is nothing to "
        "parse. Re-run cell 3 then cell 4, and let the bot finish (do not quit "
        "the loop early).\n\nLast reply was:\n" + response
    )

tail = response.split("RECAP:", 1)[1]
start = tail.find("{")
if start == -1:
    raise ValueError("Found RECAP: but no JSON object after it:\n" + tail)

# raw_decode reads one JSON object and stops, so code fences and the trailing
# READY are both ignored.
recap, _ = json.JSONDecoder().raw_decode(tail[start:])

n_days = recap["days"]
nights = recap["nights_per_city"]
transport = recap["inter_city_transport"]
country = recap.get("country")  # disambiguates the geocoder in cell 5.5
print(recap)

In [ ]:
# cell 5.5 
import requests
from babel.numbers import get_territory_currencies

MONTH_NUM = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july",
     "august", "september", "october", "november", "december"], start=1)}

CLIMATE_YEARS = (2021, 2025)  


def geocode(city, country=None):
    """City name -> location dict, via Open-Meteo geocoding.

    `country` matters: searching "Galicia" with count=1 returns a village of 416
    people in Chiapas, Mexico. We pull several hits, keep only those in the
    traveler's country, and take the largest."""
    r = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                     params={"name": city, "count": 10}, timeout=20)
    r.raise_for_status()
    hits = r.json().get("results") or []
    if not hits:
        raise ValueError(f"could not geocode {city!r}")

    if country:
        want = country.strip().lower()
        in_country = [h for h in hits
                      if (h.get("country") or "").lower() == want
                      or (h.get("country_code") or "").lower() == want]
        if not in_country:
            found = ", ".join(sorted({h.get("country") or "?" for h in hits}))
            raise ValueError(
                f"{city!r} was not found in {country!r} (only in: {found}). "
                f"Is it a region rather than a city? Re-run cell 4 and give a "
                f"city or town to sleep in."
            )
        hits = in_country

    hits.sort(key=lambda h: h.get("population") or 0, reverse=True)
    h = hits[0]
    return {"name": h["name"], "lat": h["latitude"], "lon": h["longitude"],
            "country": h.get("country"), "country_code": h.get("country_code")}


def get_weather(city, month, country=None):
    """Typical weather for `month` in `city`, averaged over CLIMATE_YEARS of real
    daily records (Open-Meteo archive). This is climate data, not a forecast:
    forecasts only reach ~16 days ahead and these trips are months away."""
    mnum = MONTH_NUM[month.strip().lower()]
    loc = geocode(city, country)
    y0, y1 = CLIMATE_YEARS
    r = requests.get("https://archive-api.open-meteo.com/v1/archive",
                     params={"latitude": loc["lat"], "longitude": loc["lon"],
                             "start_date": f"{y0}-01-01", "end_date": f"{y1}-12-31",
                             "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                             "timezone": "auto"}, timeout=60)
    r.raise_for_status()
    d = r.json()["daily"]
    rows = [(mx, mn, pr) for t, mx, mn, pr in zip(
        d["time"], d["temperature_2m_max"], d["temperature_2m_min"], d["precipitation_sum"])
        if int(t[5:7]) == mnum and None not in (mx, mn, pr)]
    if not rows:
        raise ValueError(f"no archive data for {city} in {month}")
    return {
        "city": loc["name"],
        "month": month,
        "avg_high_c": round(sum(x[0] for x in rows) / len(rows), 1),
        "avg_low_c": round(sum(x[1] for x in rows) / len(rows), 1),
        "rainy_days_in_month": round(sum(1 for x in rows if x[2] >= 1.0) / (y1 - y0 + 1), 1),
        "source": f"Open-Meteo archive, {y0}-{y1} average",
    }


def get_exchange_rate(country_code):
    """Live local-currency -> EUR rate, via frankfurter.app."""
    codes = get_territory_currencies(country_code)
    if not codes:
        return {"error": f"no currency known for {country_code}"}
    cur = codes[0]
    if cur == "EUR":
        return {"currency": "EUR", "eur_per_unit": 1.0,
                "note": "destination already uses EUR, no conversion needed"}
    r = requests.get("https://api.frankfurter.app/latest",
                     params={"from": cur, "to": "EUR"}, timeout=20)
    if r.status_code != 200 or "EUR" not in r.json().get("rates", {}):
        return {"currency": cur, "eur_per_unit": None,
                "note": f"frankfurter has no EUR rate for {cur}"}
    j = r.json()
    return {"currency": cur, "eur_per_unit": j["rates"]["EUR"], "as_of": j["date"]}


def web_search(query, model=PLANNER_MODEL):
    """Let the model search the live web. Token usage is recorded in USAGE."""
    r = client.responses.create(model=model, tools=[{"type": "web_search"}], input=query)
    USAGE.append({"model": model, "input": r.usage.input_tokens,
                  "output": r.usage.output_tokens})
    return r.output_text.strip()


def get_transport(city_a, city_b, mode):
    """Real journey time/cost between two cities, via web search."""
    if mode.strip().lower() in ("car", "drive", "driving"):
        query = (f"How far is it to drive from {city_a} to {city_b}, how long does the "
                 f"drive take, and are there tolls? Answer in two sentences.")
    else:
        query = (f"How long does the {mode} journey from {city_a} to {city_b} take door "
                 f"to door, and roughly what does a one-way ticket cost? "
                 f"Answer in two sentences.")
    return web_search(query)


def gather_facts(recap):
    """Call every tool once, so cell 6 can plan against real data."""
    cities = list(recap["nights_per_city"])
    facts = {"weather": {}, "currency": None, "transport": None}

    ctry = recap.get("country")

    for c in cities:
        try:
            facts["weather"][c] = get_weather(c, recap["month"], ctry)
        except Exception as e:
            facts["weather"][c] = {"error": f"{type(e).__name__}: {e}"}

    try:
        facts["currency"] = get_exchange_rate(
            geocode(cities[0], ctry)["country_code"])
    except Exception as e:
        facts["currency"] = {"error": f"{type(e).__name__}: {e}"}

    if len(cities) > 1:
        try:
            facts["transport"] = get_transport(cities[0], cities[1],
                                               recap["inter_city_transport"])
        except Exception as e:
            facts["transport"] = {"error": f"{type(e).__name__}: {e}"}

    return facts


facts = gather_facts(recap)
print(json.dumps(facts, indent=2, ensure_ascii=False))

In [ ]:
# cell 6

day_plan = []
for _city, _n in nights.items():
    day_plan.extend([_city] * _n)
if len(day_plan) != n_days:
    print(f"WARNING: nights sum to {len(day_plan)} but the trip is {n_days} days")
day_plan_text = "\n".join(f"Day {i}: {c}" for i, c in enumerate(day_plan, 1))
print(day_plan_text, "\n")

itinerary_prompt = f"""Write a travel itinerary for this traveler:

{json.dumps(recap, indent=2)}

Real external data retrieved for this trip (weather from historical records,
exchange rate live today, transport from a web search). Treat it as fact and
never contradict it:

{json.dumps(facts, indent=2, ensure_ascii=False)}

Day-to-city allocation. This is fixed and already decided - follow it EXACTLY,
writing one ### Day block per line below, in this order:

{day_plan_text}

Rules:
- Start with a trip title (# heading) and a two-line trip summary.
- Output exactly {n_days} ### Day blocks, matching the allocation above. Do not
  re-balance it, shorten it or extend it.
- The first day in a new city is the travel day: the inter-city journey occupies
  a named block on it, using the real duration from the retrieved transport data
  (e.g. "Morning: train London-Oxford, ~1h").
- The traveler travels between cities by {transport}. Use that mode, never another.
- The <city> in the day header must be the single city named in the allocation
  above. Never "A to B".
- Never place activities in two cities on the same day.
- Activities within one day must be geographically coherent. Anything more than
  ~45 min from the city must take the whole day, not a morning slot.
- Use the retrieved weather when planning. If the month is cold or wet, favour
  indoor activities and warn about it; if mild, favour outdoor ones. Do not
  invent temperatures - only use the figures given above.
- Only name a restaurant, venue or park you are confident actually exists. If
  unsure, describe the type of place and the street/area instead of inventing a
  name. Never invent a plausible-sounding name, and never turn a local dish into
  a restaurant name.
- Every day block must contain all five lines. Never omit a line.

Format each day exactly like this:

### Day N - <city> - <theme>
- Morning: <activity>
- Lunch: <restaurant>, <neighbourhood> - <why it fits their preferences>
- Afternoon: <activity>
- Dinner: <restaurant>, <neighbourhood> - <why it fits>
- Evening: <activity or "free">

Output the day blocks back to back, with nothing in between them. Do NOT put
accommodation, budget or tips inside a day block.

AFTER the final day block, output these four sections ONCE for the whole trip.
Each section heading must be a ### heading - the SAME markdown level as the day
headings above - with the emoji shown at both the start and the end. Copy these
four headings exactly, character for character, emoji included:

### 🏨 Accommodation 🏨
Exactly {len(nights)} lines - one per city, in this format:
<district>, <city> - <why it suits this traveler>

Hard requirements for <district>:
- It must be the PROPER NAME of a real barrio/neighbourhood of that city, the
  name a local would use and a name that appears on a map of that city.
  Good: "Juderia, Segovia". Bad: "Segovia Old Town, Segovia".
- It is FORBIDDEN to use a generic description instead of a name. Reject:
  "centre", "city centre", "city center", "old town", "old quarter",
  "historic centre", "casco antiguo", "downtown", and any translation or
  variation of these. A district name is a proper noun, not a description.
- The district must lie INSIDE the city it is listed under - never a separate
  nearby town or municipality.
- If you are not confident that you know a real barrio name for that city, do
  NOT guess and do NOT fall back on a generic phrase. Instead name a specific
  street or landmark to stay beside, e.g. "beside the Aqueduct, Calle Cervantes".
- The district must not change while the traveler is in the same city.

### 🌦️ Weather 🌦️
One line per city, quoting the retrieved figures exactly:
<city>: typical <month> high <X>C / low <Y>C, about <Z> rainy days that month.

### 💰 Estimated budget 💰
Estimate it yourself from typical mid-range prices for {recap['month']}. Never
ask the traveler for a budget. Use the local currency named in the retrieved
currency data, on every line. Output exactly these five lines, replacing CUR
with that currency code:
- Lodging: CUR X-Y
- Food: CUR X-Y
- Activities: CUR X-Y
- Transport: CUR X-Y
- **Total: CUR X-Y**
All figures per person, for the WHOLE TRIP - not per day, not per night. Use
whole numbers only. The total must be the sum of the four lines above - add them
up and check before writing.
Then, ONLY if the retrieved currency is not EUR and eur_per_unit is not null,
add one more line converting the total using that exact rate, rounded to whole
euros:
- **Total in EUR: EUR X-Y** (at 1 CUR = <eur_per_unit> EUR, <as_of>)

### 💡 Practical tips 💡
3-5 bullets. At least one must be about what to pack for the weather above.
"""

planner_messages = [
    {'role': 'system', 'content': """You are an expert travel planner. You write
complete, realistic, day-by-day itineraries. You never ask questions - you have
all the information you need. Output the full itinerary in one message."""},
    {'role': 'user', 'content': itinerary_prompt},
]

itinerary = get_completion_from_messages(planner_messages, model=PLANNER_MODEL, temperature=0.7)
print(itinerary)

In [ ]:
# cell 7
summary_prompt = """Convert the itinerary you just wrote into JSON.

Copy the details exactly as written above — do not invent, re-plan, or omit
anything. If a field is missing from the itinerary, use null.

Return only the JSON object, with this shape:
{
  "trip_title": str, "trip_summary": str, "accommodation": str,
"days": [ {"day": int, "city": str, "theme": str,
           "eat": {"lunch": str, "dinner": str},
           "do": [str]} ],
  
  "tips": [str],
  "estimated_budget": {"lodging": str, "food": str,
                       "activities": str, "transport": str, "total": str}
}
"""

summary = get_completion_from_messages(
    [{'role': 'user', 'content': itinerary + "\n\n" + summary_prompt}],
    model=PLANNER_MODEL,
    temperature=0,
)


_start = summary.find("{")
if _start == -1:
    raise ValueError("No JSON object in the summary reply:\n" + summary)
summary_data, _ = json.JSONDecoder().raw_decode(summary[_start:])

print(json.dumps(summary_data, indent=2, ensure_ascii=False))

In [ ]:
# cell 7.5 - one big poster for the trip, one small image per day
import base64
from pathlib import Path

IMAGE_DIR = Path("trip_images")
IMAGE_DIR.mkdir(exist_ok=True)

STYLE = ("Vintage travel poster illustration, flat shapes, warm limited palette, "
         "clean composition. No text, no lettering, no words, no captions anywhere.")


def generate_image(prompt, filename, size="1024x1024", quality=None):
    """Make one image, record its token usage, save it, return the path."""
    r = client.images.generate(model=IMAGE_MODEL, prompt=prompt, size=size,
                               quality=quality or IMAGE_QUALITY, n=1)
    USAGE.append({"model": IMAGE_MODEL,
                  "input": r.usage.input_tokens,
                  "output": r.usage.output_tokens})
    path = IMAGE_DIR / filename
    path.write_bytes(base64.b64decode(r.data[0].b64_json))
    return path


def _interests_text(recap):
    i = recap.get("interests")
    return ", ".join(i) if isinstance(i, list) else str(i)


def hero_image_prompt(recap, facts):
    cities = list(recap["nights_per_city"])
    return (f"{STYLE} Subject: a single poster for one trip visiting "
            f"{' and '.join(cities)}, {recap.get('country', '')} in "
            f"{recap['month']}. Combine their most recognisable real landmarks "
            f"into one scene. Hint at these interests: {_interests_text(recap)}.")


def day_image_prompt(day, recap):
    """One image for one day, driven by what that day actually does."""
    activities = "; ".join(day.get("do") or [])[:280]
    return (f"{STYLE} One single clear scene, not a collage. "
            f"Subject: {day.get('theme')} in {day.get('city')}, "
            f"{recap.get('country', '')}, in {recap['month']}. "
            f"Depict this day's activities: {activities}")


days = summary_data["days"]
n_imgs = len(days) + 1
print(f"Generating {n_imgs} images: 1 hero ({IMAGE_QUALITY}) + {len(days)} daily "
      f"({DAY_IMAGE_QUALITY}), ~${0.042 + 0.011 * len(days):.2f}. "
      f"About {n_imgs * 19}s...")

hero_path = generate_image(hero_image_prompt(recap, facts), "00_hero.png",
                           quality=IMAGE_QUALITY)
print("  hero ->", hero_path)

day_images = {}
for d in days:
    num = d["day"]
    day_images[num] = generate_image(day_image_prompt(d, recap),
                                     f"day_{num:02d}.png",
                                     quality=DAY_IMAGE_QUALITY)
    print(f"  day {num} ({d.get('theme')}) -> {day_images[num]}")

In [ ]:
# cell 8 - 
EUR_PER_USD = 0.92  


def usage_report(usage=USAGE):
    if not usage:
        print("No API calls recorded. Run cells 3-7 first.")
        return

    by_model = {}
    for rec in usage:
        m = by_model.setdefault(rec["model"], {"calls": 0, "input": 0, "output": 0})
        m["calls"] += 1
        m["input"] += rec["input"]
        m["output"] += rec["output"]

    print(f"{'model':<16}{'calls':>6}{'input':>9}{'output':>9}{'cost $':>10}")
    print("-" * 50)

    total_cost = 0.0
    for model, m in by_model.items():
        price = PRICES.get(model)
        if price is None:
            cost_str = "no price"
        else:
            cost = (m["input"] / 1e6 * price["input"]
                    + m["output"] / 1e6 * price["output"])
            total_cost += cost
            cost_str = f"{cost:.4f}"
        print(f"{model:<16}{m['calls']:>6}{m['input']:>9,}{m['output']:>9,}{cost_str:>10}")

    tot_in = sum(m["input"] for m in by_model.values())
    tot_out = sum(m["output"] for m in by_model.values())
    print("-" * 50)
    print(f"{'TOTAL':<16}{len(usage):>6}{tot_in:>9,}{tot_out:>9,}{total_cost:>10.4f}")
    print(f"\nTokens: {tot_in:,} in + {tot_out:,} out = {tot_in + tot_out:,}")
    print(f"Estimated cost: ${total_cost:.4f}  (EUR {total_cost * EUR_PER_USD:.4f})")


usage_report()

In [ ]:
# cell 9 - the finished trip, formatted for a human to read
import re
from IPython.display import display, Markdown, Image as IPyImage


def render_trip():
    # --- title, summary, poster for the whole trip
    display(Markdown(f"# \U0001F30D {summary_data['trip_title']} \U0001F30D"))
    display(Markdown(f"*{summary_data['trip_summary']}*"))
    display(IPyImage(filename=str(hero_path), width=620))

    # --- split into day blocks + the trailing sections.
    # The tail starts at the first ### heading that is not a day.
    first_day = itinerary.find("### Day")
    rest = itinerary[first_day:] if first_day != -1 else itinerary
    m = re.search(r"^###\s+(?!Day\b)", rest, flags=re.M)
    body, tail = (rest[:m.start()], rest[m.start():]) if m else (rest, "")

    blocks = [b for b in re.split(r"(?=^### Day )", body, flags=re.M) if b.strip()]

    display(Markdown("---\n## \U0001F5D3️ Day by day \U0001F5D3️"))
    for block in blocks:
        lines = block.rstrip().splitlines()
        header, detail = lines[0], "\n".join(lines[1:])

        display(Markdown(header))
        num = re.match(r"###\s+Day\s+(\d+)", header)
        if num and int(num.group(1)) in day_images:
            display(IPyImage(filename=str(day_images[int(num.group(1))]), width=300))
        display(Markdown(detail))

    # --- accommodation / weather / budget / tips, exactly as written
    display(Markdown("---\n" + tail))

    # --- what it cost to build
    display(Markdown("---\n### \U0001F9FE Token usage and cost \U0001F9FE"))
    usage_report()


render_trip()